# WORK9 — Stage 5.1 Underforecast / Demand-Gap Diagnosis V0.1

Diagnosis only. **No model retraining, no Frozen Test.**

This notebook explains where LightGBM underforecast comes from and tests M² output rounding policies.


In [1]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install pyarrow pyyaml

from pathlib import Path
import json, sys, subprocess
import pandas as pd

ROOT = Path('/content/drive/MyDrive/work9')
SRC = ROOT / '02_src/modeling/underforecast_diagnosis_runner_v01.py'
TEST = ROOT / '07_tests/test_underforecast_diagnosis_runner_v01.py'
CONTRACT = ROOT / '01_config/underforecast_diagnosis_contract_v01.yaml'
ROLL_PTR = ROOT / '01_config/current_rolling_backtest_run.json'

for p in [SRC, TEST, CONTRACT, ROLL_PTR]:
    assert p.exists(), f'Missing: {p}'
print('Inputs found')


Mounted at /content/drive
Inputs found


## 1. Verify current rolling run


In [2]:
rolling = json.loads(ROLL_PTR.read_text(encoding='utf-8'))
print(json.dumps(rolling, indent=2, ensure_ascii=False))
assert rolling['status'] == 'PASS'
assert rolling['backtest_version'] == 'rolling_backtest_v01'

report_dir = Path(rolling['report_dir'])
pred_path = report_dir / 'rolling_origin_predictions.parquet'
assert pred_path.exists(), pred_path
print('Current rolling predictions:', pred_path)


{
  "run_id": "rolling_backtest_v01_20260815T132319Z",
  "status": "PASS",
  "backtest_version": "rolling_backtest_v01",
  "source_model_candidate_run_id": "pair_modeling_v02_20260815T130724Z",
  "report_dir": "/content/drive/MyDrive/work9/06_reports/rolling_backtest/rolling_backtest_v01_20260815T132319Z",
  "run_manifest_path": "/content/drive/MyDrive/work9/08_runs/rolling_backtest_v01_20260815T132319Z/run_manifest.json",
  "backtest_manifest_path": "/content/drive/MyDrive/work9/06_reports/rolling_backtest/rolling_backtest_v01_20260815T132319Z/backtest_manifest.json",
  "backtest_manifest_sha256": "253a5a54d475d852364ef9ab1c5b55eac38d1ad4810141f6ebc9187d9afdf8f4",
  "created_at_utc": "2026-08-15T13:51:43.338150+00:00"
}
Current rolling predictions: /content/drive/MyDrive/work9/06_reports/rolling_backtest/rolling_backtest_v01_20260815T132319Z/rolling_origin_predictions.parquet


## 2. Unit tests


In [3]:
r = subprocess.run(
    [sys.executable, '-m', 'pytest', '-q', str(TEST)],
    text=True, capture_output=True
)
print(r.stdout)
if r.returncode != 0:
    print(r.stderr)
    raise AssertionError(f'Diagnosis unit tests failed (returncode={r.returncode})')
print('Diagnosis unit tests PASS')


.......                                                                  [100%]
7 passed in 3.72s

Diagnosis unit tests PASS


## 3. Run diagnosis


In [4]:
import importlib.util, datetime
spec = importlib.util.spec_from_file_location('diag', SRC)
diag = importlib.util.module_from_spec(spec)
spec.loader.exec_module(diag)

run_id = 'underforecast_diagnosis_v01_' + datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
out_dir = ROOT / '06_reports/underforecast_diagnosis' / run_id
manifest = diag.run_diagnosis(
    rolling_prediction_path=str(pred_path),
    rolling_pointer=rolling,
    contract_path=str(CONTRACT),
    report_dir=str(out_dir),
)
print('Diagnosis PASS:', run_id)
print(json.dumps(manifest['rounding_policy'], indent=2))


Diagnosis PASS: underforecast_diagnosis_v01_20260815T140720Z
{
  "training_target_rounded": false,
  "actual_evaluation_rounded": false,
  "forecast_rounding_is_diagnostic_only": true,
  "policies_tested": [
    "raw",
    "round_nearest_1m2",
    "ceil_1m2"
  ],
  "preferred_business_display_rule": "round_after_aggregation_if_integer_M2_display_is_required"
}


## 4. Audit underforecast attribution


In [5]:
under = pd.read_csv(out_dir / 'underforecast_diagnosis.csv')
events = under[under['dimension'].eq('DEMAND_EVENT')].sort_values('share_of_total_underforecast', ascending=False)
display(events[['bucket','n_rows','actual_sum_m2','forecast_sum_m2','wape','bias_ratio','underforecast_m2','share_of_total_underforecast']])

print('Behavior:')
display(under[under['dimension'].eq('BEHAVIOR')][['bucket','n_rows','wape','bias_ratio','share_of_total_underforecast']])

print('Horizon:')
display(under[under['dimension'].eq('HORIZON')][['bucket','n_rows','wape','bias_ratio','share_of_total_underforecast']])


,bucket,n_rows,actual_sum_m2,forecast_sum_m2,wape,bias_ratio,underforecast_m2,share_of_total_underforecast
15,ongoing_positive,90907,9040093.34,7.006339e+06,0.665213,-0.224970,4.023669e+06,0.385495
35,spike_3x_plus,10334,2802406.50,5.241750e+05,0.818551,-0.812955,2.286072e+06,0.219021
19,reactivation_gap_1_2,34038,2818147.32,9.956244e+05,0.794498,-0.646710,2.030768e+06,0.194562
31,spike_2x_3x,8103,1852084.48,5.163316e+05,0.728083,-0.721216,1.342112e+06,0.128584
23,reactivation_gap_3_5,9711,678834.18,1.061596e+05,0.886550,-0.843615,5.872474e+05,0.056262
27,reactivation_gap_6plus,3075,174718.02,1.284738e+04,0.939454,-0.926468,1.630051e+05,0.015617
11,first_positive_known_pair,110,5646.41,1.115651e+03,0.893340,-0.802414,4.787460e+03,0.000459
7,actual_zero,264568,0.00,4.874809e+06,NaN,NaN,0.000000e+00,0.000000


Behavior:


,bucket,n_rows,wape,bias_ratio,share_of_total_underforecast
4,intermittent,289442,1.090338,-0.170919,0.559093
5,regular,51219,0.816168,-0.259418,0.308458
6,very_sparse,80185,1.203341,-0.093758,0.132449


Horizon:


,bucket,n_rows,wape,bias_ratio,share_of_total_underforecast
1,H1,140425,0.968527,-0.152887,0.344524
2,H2,140096,1.022007,-0.196367,0.330217
3,H3,140325,1.046447,-0.234496,0.325259


## 5. Audit M² granularity and rounding


In [6]:
gran = pd.read_csv(out_dir / 'actual_m2_granularity_summary.csv')
frac = pd.read_csv(out_dir / 'actual_m2_fraction_frequency.csv')
round_m = pd.read_csv(out_dir / 'rounding_pair_month_comparison.csv')
round_3m = pd.read_csv(out_dir / 'rounding_3m_comparison.csv')

print('Actual M2 granularity:')
display(gran)
print('Most common fractional parts:')
display(frac.head(20))
print('Pair-month rounding comparison:')
display(round_m[['policy','wape','bias_ratio','forecast_sum_m2','added_m2_vs_raw','changed_row_rate','raw_pred_between_0_and_1_rate']])
print('3M rounding comparison:')
display(round_3m[['level','policy','wape','bias_ratio','forecast_sum_m2']].sort_values(['level','wape']))


Actual M2 granularity:


,n_positive_actual_rows,actual_sum_m2,share_exact_integer_m2,share_on_0_1_m2_grid,share_on_0_01_m2_grid,share_on_0_001_m2_grid,min_positive_m2,p01_positive_m2,p05_positive_m2,median_positive_m2,p95_positive_m2
0,156278,17371930.25,0.114616,0.336957,1.0,1.0,0.08,0.5,1.8,39.6,402.0


Most common fractional parts:


,fractional_part_3dp,n_rows,row_rate
0,0.00,17912,0.114616
1,0.20,6508,0.041644
2,0.80,6445,0.041241
3,0.40,6097,0.039014
4,0.50,6064,0.038803
5,0.60,6011,0.038464
6,0.88,4851,0.031041
7,0.08,4822,0.030855
8,0.32,4603,0.029454
9,0.96,4560,0.029179


Pair-month rounding comparison:


,policy,wape,bias_ratio,forecast_sum_m2,added_m2_vs_raw,changed_row_rate,raw_pred_between_0_and_1_rate
0,raw,1.009720,-0.191949,1.403740e+07,0.000000,0.0,0.115399
1,round_nearest_1m2,1.009559,-0.192125,1.403435e+07,-3051.204605,1.0,0.115399
2,ceil_1m2,1.016895,-0.179595,1.425202e+07,214613.795395,1.0,0.115399


3M rounding comparison:


,level,policy,wape,bias_ratio,forecast_sum_m2
5,BASE_SKU_3M,round_nearest_1m2_before_aggregate,0.447487,-0.188696,1.342843e+07
13,BASE_SKU_3M,round_nearest_1m2_after_aggregate,0.447574,-0.188513,1.343147e+07
1,BASE_SKU_3M,raw_before_aggregate,0.447575,-0.188512,1.343149e+07
17,BASE_SKU_3M,ceil_1m2_after_aggregate,0.447639,-0.188126,1.343787e+07
9,BASE_SKU_3M,ceil_1m2_before_aggregate,0.449425,-0.175983,1.363886e+07
10,BRANCH_3M,ceil_1m2_before_aggregate,0.306647,-0.175983,1.363886e+07
18,BRANCH_3M,ceil_1m2_after_aggregate,0.309991,-0.188492,1.343182e+07
2,BRANCH_3M,raw_before_aggregate,0.309998,-0.188512,1.343149e+07
14,BRANCH_3M,round_nearest_1m2_after_aggregate,0.309999,-0.188512,1.343148e+07
6,BRANCH_3M,round_nearest_1m2_before_aggregate,0.310002,-0.188696,1.342843e+07


## 6. Write diagnosis pointer only after PASS


In [7]:
ptr = {
    'run_id': run_id,
    'status': 'PASS',
    'diagnosis_version': 'underforecast_diagnosis_v01',
    'source_rolling_run_id': rolling['run_id'],
    'report_dir': str(out_dir),
    'diagnosis_manifest_path': str(out_dir / 'diagnosis_manifest.json'),
}
ptr_path = ROOT / '01_config/current_underforecast_diagnosis_run.json'
ptr_path.write_text(json.dumps(ptr, ensure_ascii=False, indent=2), encoding='utf-8')
print('UNDERFORECAST DIAGNOSIS PASS')
print('Pointer:', ptr_path)
print('STOP HERE. Send Work9 diagnosis result for audit before any freeze/test decision.')


UNDERFORECAST DIAGNOSIS PASS
Pointer: /content/drive/MyDrive/work9/01_config/current_underforecast_diagnosis_run.json
STOP HERE. Send Work9 diagnosis result for audit before any freeze/test decision.
